# 19 — End-to-End NISAR Polarimetric Analysis Capstone

Bring one HDF5 product and let the automatic routing do its job — the same spatial_subset from Module 06 carries through the whole thing.

- Full-pol product → C3, eigen analysis, H/A/α, Pauli, Freeman–Durden, Yamaguchi
- Compact/hybrid product → Stokes parameters, m-χ, m-δ, m-α

No second AOI, no manual branch selection.

In [ ]:
import numpy as np
from nisar_utils.config import load_config
from nisar_utils.polarimetric_io import *
from nisar_utils.polarimetry import *
cfg=load_config(); mode=load_detected_mode(cfg); sp=get_authoritative_subset(cfg)
print('Product:',cfg['nisar_file'])
print('Authoritative persisted subset:',sp)
print('Automatic branch:',mode['polarimetric_mode'])
if mode['polarimetric_mode']=='LSAR_FULL_POL':
    data,sp,guard,_=load_branch_terms(cfg,FULL_POL_TERMS); Craw=covariance_hermitian_from_terms(data); Cstd=nisar_c3_to_standard_c3(Craw); T3=standard_c3_to_t3(Cstd); e=eigen_analysis(T3)
    alpha=cloude_pottier_mean_alpha(e['eigenvectors'],e['probabilities'])*180/np.pi
    pauli=pauli_powers(Craw); fd=fullpol_freeman_durden(Craw); y4=fullpol_yamaguchi4(Craw)
    print('FULL-POL results:')
    print('  Span median:',float(np.nanmedian(e['span'])))
    print('  H median:',float(np.nanmedian(e['entropy'])))
    print('  A median:',float(np.nanmedian(e['anisotropy'])))
    print('  Mean alpha median (deg):',float(np.nanmedian(alpha)))
    print('  Freeman residual RMS:',float(np.sqrt(np.nanmean(fd['residual']**2))))
    print('  Yamaguchi residual RMS:',float(np.sqrt(np.nanmean(y4['residual']**2))))
elif mode['polarimetric_mode']=='SSAR_COMPACT_POL':
    data,sp,guard,_=load_branch_terms(cfg,COMPACT_POL_TERMS); g0,g1,g2,g3=stokes_from_compact_terms(data['RHRH'],data['RHRV'],data['RVRV']); p=compact_child_parameters(g0,g1,g2,g3)
    for n,d in [('m-chi',m_chi_decomposition(g0,p['m'],p['chi2'])),('m-delta',m_delta_decomposition(g0,p['m'],p['delta'])),('m-alpha',m_alpha_decomposition(g0,p['m'],p['alpha_s']))]:
        print(n,'median residual:',float(np.nanmedian(decomposition_residual(g0,d))))
    print('COMPACT-POL results: m=',float(np.nanmedian(p['m'])),'delta(deg)=',float(np.nanmedian(np.degrees(p['delta']))),'alpha_s(deg)=',float(np.nanmedian(np.degrees(p['alpha_s']))))
else:
    raise RuntimeError('Unsupported polarimetric configuration for this current training branch.')